In [7]:
import geopandas as gpd
import pandas as pd

In [9]:
oan = gpd.read_file("../data/data/oan/oan_registros_clean.geojson")
chla_gems = gpd.read_file("../data/data/joins/gems_chla.geojson")
gem_puntos_examin = pd.read_csv("../data/data/puntos_examinados.csv")

In [10]:
gem_puntos_examin.head()

,nombre,Decision,Fecha,Width,Resolution,geometry
0,URY00006,si,2025-02-13 13:52:11.656000,0.05,10,POINT (-55.3727 -34.5071)
1,URY00007,dudoso,2025-02-13 13:51:57.401000,0.05,10,POINT (-55.2762 -34.391)
2,URY00008,dudoso,2025-02-13 13:51:57.401000,0.05,10,POINT (-55.2061 -34.4204)
3,URY00009,dudoso,2025-02-13 13:51:57.401000,0.05,10,POINT (-55.2471 -34.3791)
4,URY00010,dudoso,2025-02-13 13:51:57.401000,0.05,10,POINT (-55.2317 -34.3828)


In [41]:
chla_gems = chla_gems[pd.to_datetime(chla_gems["fecha"]).dt.year >= 2017]
chla_gems["fecha"].min()

Timestamp('2017-01-02 00:00:00')

In [42]:
oan_utm  = oan.to_crs(32721).copy()
gems_utm = chla_gems.to_crs(32721).copy()

In [43]:
#oan_est  = oan_utm[['x','y','id_estacion']].drop_duplicates()
#gems_est = gems_utm[['x','y','estacion' ]].drop_duplicates()
#
#pares = gems_est.merge(oan_est, on=['x','y'], how='inner')
#
#print(len(pares), 'estaciones que coinciden')

In [44]:
oan_utm.head()

,nombre_programa,estacion,id_estacion,nro_muestra,departamento,fecha_hora,nombre_clave,uni_nombre,valor_original,limite_deteccion,limite_cuantificacion,valor_transformado,geometry
0,Agua Arroyo Grande del Norte DCA,XGRN100.S,100626,29873.0,RÍO NEGRO,2019-09-05 16:00:00,CloA_(lab),µg/L,LD<x<LC,0.700000000,2.2,LD<x<LC,POINT (461374.621 6359080.918)
1,Agua Arroyo Grande del Norte DCA,XGRN100.S,100626,29985.0,RÍO NEGRO,2019-10-31 09:40:00,CloA_(lab),µg/L,LD<x<LC,0.700000000,2.2,LD<x<LC,POINT (461374.621 6359080.918)
2,Agua Arroyo Grande del Norte DCA,XGRN100.S,100626,31062.0,RÍO NEGRO,2020-06-04 14:20:00,CloA_(lab),µg/L,LD<X<LC,0.700000000,2.2,LD<X<LC,POINT (461374.621 6359080.918)
3,Agua Arroyo Grande del Norte DCA,XGRN100.S,100626,31410.0,RÍO NEGRO,2020-08-06 12:46:00,CloA_(lab),µg/L,33.000000000,0.700000000,2.2,33.000000000,POINT (461374.621 6359080.918)
4,Agua Arroyo Grande del Norte DCA,XGRN100.S,100626,31851.0,RÍO NEGRO,2020-11-19 12:29:00,CloA_(lab),µg/L,2.600000000,0.700000000,2.2,2.600000000,POINT (461374.621 6359080.918)


In [45]:
iguales = gpd.sjoin_nearest(
    gems_utm, oan_utm,
    how='inner',
    max_distance=0.1,
    distance_col='dist'
)

In [91]:
gems_stations = gems_utm["geometry"].unique()
oan_stations = oan_utm["geometry"].unique()

In [100]:
gems_stations = gpd.GeoDataFrame(
    geometry=gems_utm["geometry"].unique(),
    crs=gems_utm.crs,
)
oan_stations = gpd.GeoDataFrame(
    geometry = oan_utm["geometry"].unique(),
    crs = oan_utm.crs
)

In [101]:
iguales = gpd.sjoin_nearest(
    gems_stations, oan_stations,
    how='inner',
    max_distance=0.1,       # in the CRS units — meters here, since you're in UTM
    distance_col='dist'
)
iguales = iguales.drop(columns=["index_right", "dist"])

In [103]:
iguales.shape

(150, 1)

Esto nos dice que tanto en los registros de OAN como en los de Gems, existen estaciones que son identicas.

Lo que no significa que sean las mismas mediciones ni los mismos dias, eso es algo que tenemos que comprobar todavia.

Pero puede ser que estos datos sean compartidos entre ambas organizaciones.

In [104]:
iguales.head(5)

,geometry
0,POINT (649386.009 6180413.463)
1,POINT (664852.016 6189770.245)
2,POINT (570470.059 6220119.303)
3,POINT (581348.445 6603897.643)
4,POINT (560699.232 6625269.716)


In [105]:
#4326
#oan_coords = oan_utm.to_crs(4326)

In [106]:
iguales_wgs84 = iguales.to_crs(4326).round(4)

iguales_wgs84.head(5)

,geometry
0,POINT (-55.3727 -34.5071)
1,POINT (-55.2061 -34.4204)
2,POINT (-56.2355 -34.15747)
3,POINT (-56.15059 -30.69494)
4,POINT (-56.36744 -30.50333)


In [107]:
#p1 = iguales_wgs84.geometry.iloc[0]

In [108]:
#p1.x, p1.y

In [109]:
gems_utm.head()

,estacion,Decision,param,fecha,value,unit,depth,granularidad,geometry
0,URY00006,si,Chl-a,2018-11-12 23:20:00,0.0000,mg/l,0.3,DIA,POINT (649386.009 6180413.463)
1,URY00008,dudoso,Chl-a,2018-11-12 08:30:00,0.0000,mg/l,0.3,DIA,POINT (664852.016 6189770.245)
6,URY00029,si,Chl-a,2017-02-22 11:00:00,0.0004,mg/l,0.3,DIA,POINT (570470.059 6220119.303)
7,URY00029,si,Chl-a,2017-04-19 13:00:00,0.0007,mg/l,0.3,DIA,POINT (570470.059 6220119.303)
8,URY00029,si,Chl-a,2017-12-13 11:44:00,0.0286,mg/l,0.3,DIA,POINT (570470.059 6220119.303)


In [110]:
p1 = iguales.iloc[[0]]
p1

,geometry
0,POINT (649386.009 6180413.463)


In [114]:
chla_p1_gems = gpd.sjoin_nearest(
    p1, gems_utm,
    how='inner',
    max_distance=0.1,       # in the CRS units — meters here, since you're in UTM
    distance_col='dist'
)

In [115]:
chla_p1_gems.head()

,geometry,index_right,estacion,Decision,param,fecha,value,unit,depth,granularidad,dist
0,POINT (649386.009 6180413.463),0,URY00006,si,Chl-a,2018-11-12 23:20:00,0.0,mg/l,0.3,DIA,0.0


In [118]:
OAN_p1_chla = gpd.sjoin_nearest(
    p1, oan_utm,
    how='inner',
    max_distance=0.1,       # in the CRS units — meters here, since you're in UTM
    distance_col='dist'
)

In [119]:
OAN_p1_chla.head()

,geometry,index_right,nombre_programa,estacion,id_estacion,nro_muestra,departamento,fecha_hora,nombre_clave,uni_nombre,valor_original,limite_deteccion,limite_cuantificacion,valor_transformado,dist
0,POINT (649386.009 6180413.463),5149,Playa,08AB,100012,1.0,LAVALLEJA,2018-11-12 23:20:00,CloA_(lab),µg/L,0.000,NaN,NaN,0.000,0.0


In [120]:
def examinar_punto(p):
    chla_p_gems = gpd.sjoin_nearest(
        p, gems_utm,
        how='inner',
        max_distance=0.1,       # in the CRS units — meters here, since you're in UTM
        distance_col='dist'
    ) 

    OAN_p_chla = gpd.sjoin_nearest(
        p, oan_utm,
        how='inner',
        max_distance=0.1,       # in the CRS units — meters here, since you're in UTM
        distance_col='dist'
    )
    
    display(chla_p_gems)
    display(OAN_p_chla)

In [121]:
examinar_punto(p1)

,geometry,index_right,estacion,Decision,param,fecha,value,unit,depth,granularidad,dist
0,POINT (649386.009 6180413.463),0,URY00006,si,Chl-a,2018-11-12 23:20:00,0.0,mg/l,0.3,DIA,0.0


,geometry,index_right,nombre_programa,estacion,id_estacion,nro_muestra,departamento,fecha_hora,nombre_clave,uni_nombre,valor_original,limite_deteccion,limite_cuantificacion,valor_transformado,dist
0,POINT (649386.009 6180413.463),5149,Playa,08AB,100012,1.0,LAVALLEJA,2018-11-12 23:20:00,CloA_(lab),µg/L,0.000,NaN,NaN,0.000,0.0


In [123]:
p2 = iguales.iloc[[1]]
examinar_punto(p2)

,geometry,index_right,estacion,Decision,param,fecha,value,unit,depth,granularidad,dist
1,POINT (664852.016 6189770.245),1,URY00008,dudoso,Chl-a,2018-11-12 08:30:00,0.0,mg/l,0.3,DIA,0.0


,geometry,index_right,nombre_programa,estacion,id_estacion,nro_muestra,departamento,fecha_hora,nombre_clave,uni_nombre,valor_original,limite_deteccion,limite_cuantificacion,valor_transformado,dist
1,POINT (664852.016 6189770.245),5121,Playa,06SF,100014,0.0,LAVALLEJA,2018-11-12 08:30:00,CloA_(lab),µg/L,0.000,NaN,NaN,0.000,0.0


In [126]:
p3 = iguales.iloc[[2]]
examinar_punto(p3)

,geometry,index_right,estacion,Decision,param,fecha,value,unit,depth,granularidad,dist
2,POINT (570470.059 6220119.303),6,URY00029,si,Chl-a,2017-02-22 11:00:00,0.0004,mg/l,0.3,DIA,0.0
2,POINT (570470.059 6220119.303),7,URY00029,si,Chl-a,2017-04-19 13:00:00,0.0007,mg/l,0.3,DIA,0.0
2,POINT (570470.059 6220119.303),8,URY00029,si,Chl-a,2017-12-13 11:44:00,0.0286,mg/l,0.3,DIA,0.0
2,POINT (570470.059 6220119.303),9,URY00029,si,Chl-a,2018-02-21 11:15:00,0.0049,mg/l,0.3,DIA,0.0
2,POINT (570470.059 6220119.303),10,URY00029,si,Chl-a,2018-10-17 11:43:00,0.0025,mg/l,0.3,DIA,0.0
2,POINT (570470.059 6220119.303),11,URY00029,si,Chl-a,2019-04-24 11:00:00,0.0023,mg/l,0.3,DIA,0.0
2,POINT (570470.059 6220119.303),20,URY00029,si,Chl-a,2022-10-19 11:26:00,0.0071,mg/l,0.3,DIA,0.0
2,POINT (570470.059 6220119.303),19,URY00029,si,Chl-a,2022-08-31 11:09:00,0.0096,mg/l,0.3,DIA,0.0
2,POINT (570470.059 6220119.303),18,URY00029,si,Chl-a,2022-02-22 13:37:00,0.0067,mg/l,0.3,DIA,0.0
2,POINT (570470.059 6220119.303),17,URY00029,si,Chl-a,2021-12-15 11:00:00,0.0740,mg/l,0.3,DIA,0.0


,geometry,index_right,nombre_programa,estacion,id_estacion,nro_muestra,departamento,fecha_hora,nombre_clave,uni_nombre,valor_original,limite_deteccion,limite_cuantificacion,valor_transformado,dist
2,POINT (570470.059 6220119.303),1734,Agua rio Santa Lucia DCA,XSLH030.S,100201,26596.0,FLORIDA,2017-10-18 11:58:00,CloA_(lab),µg/L,<LC,0.600,1.5,<LC,0.0
2,POINT (570470.059 6220119.303),1733,Agua rio Santa Lucia DCA,XSLH030.S,100201,26124.0,FLORIDA,2017-06-21 11:05:00,CloA_(lab),µg/L,<LD,0.600,1.5,<LD,0.0
2,POINT (570470.059 6220119.303),1745,Agua rio Santa Lucia DCA,XSLH030.S,100201,29435.0,FLORIDA,2019-06-12 11:15:00,CloA_(lab),µg/L,<LD,0.700,2.2,<LD,0.0
2,POINT (570470.059 6220119.303),1771,Agua rio Santa Lucia DCA,XSLH030.S,100201,37542.0,FLORIDA,2024-06-04 12:19:00,CloA_(lab),µg/L,LD<X<LC,0.700000000,2.2,LD<X<LC,0.0
2,POINT (570470.059 6220119.303),1772,Agua rio Santa Lucia DCA,XSLH030.S,100201,37767.0,FLORIDA,2024-08-13 10:41:00,CloA_(lab),µg/L,3.500000000,0.700000000,2.2,3.500000000,0.0
2,POINT (570470.059 6220119.303),1773,Agua rio Santa Lucia DCA,XSLH030.S,100201,38119.0,FLORIDA,2024-10-22 12:00:00,CloA_(lab),µg/L,11.000000000,0.700000000,2.2,11.000000000,0.0
2,POINT (570470.059 6220119.303),1774,Agua rio Santa Lucia DCA,XSLH030.S,100201,38432.0,FLORIDA,2024-12-10 11:49:00,CloA_(lab),µg/L,2.400000000,0.700000000,2.2,2.400000000,0.0
2,POINT (570470.059 6220119.303),1769,Agua rio Santa Lucia DCA,XSLH030.S,100201,36953.0,FLORIDA,2024-02-06 11:56:00,CloA_(lab),µg/L,7.900000000,0.700000000,2.2,7.900000000,0.0
2,POINT (570470.059 6220119.303),1753,Agua rio Santa Lucia DCA,XSLH030.S,100201,31429.0,FLORIDA,2020-08-12 11:42:00,CloA_(lab),µg/L,2.800000000,0.700000000,2.2,2.800000000,0.0
2,POINT (570470.059 6220119.303),1767,Agua rio Santa Lucia DCA,XSLH030.S,100201,36744.0,FLORIDA,2023-12-13 09:20:00,CloA_(lab),µg/L,2.200000000,0.700000000,2.2,2.200000000,0.0
